# Lesson 1b: MDP Practical - Dynamic Programming Foundations

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement Policy Evaluation** - Compute state values for a given policy
2. **Implement Policy Iteration** - Find optimal policies through iterative improvement
3. **Implement Value Iteration** - Combine evaluation and improvement in one step
4. **Apply to Real Environments** - Solve FrozenLake and GridWorld
5. **Visualize Solutions** - Plot value functions and optimal policies
6. **Compare Methods** - Analyze convergence and computational efficiency

This notebook builds on the theoretical foundations from Lesson 1a and provides practical implementations of dynamic programming methods for solving MDPs.

---

## Table of Contents

1. [Setup and Imports](#setup)
2. [Policy Evaluation](#policy-evaluation)
3. [Policy Iteration](#policy-iteration)
4. [Value Iteration](#value-iteration)
5. [FrozenLake Environment](#frozenlake)
6. [GridWorld Solutions](#gridworld)
7. [Visualization and Analysis](#visualization)
8. [Performance Comparison](#comparison)
9. [Exercises](#exercises)

---

## 1. Setup and Imports <a id='setup'></a>

First, let's import the necessary libraries and set up our environment.

In [ ]:
# Standard library imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List, Optional, Callable
from dataclasses import dataclass
from collections import defaultdict
import time

# Gymnasium for RL environments
import gymnasium as gym

# Set random seed for reproducibility
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Gymnasium version: {gym.__version__}")

## 2. Policy Evaluation <a id='policy-evaluation'></a>

### Theory Recap

Policy evaluation computes the state-value function $V^\pi(s)$ for a given policy $\pi$.

**Bellman Expectation Equation:**

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a) [r + \gamma V^\pi(s')]$$

**Iterative Policy Evaluation Algorithm:**

1. Initialize $V(s) = 0$ for all states
2. Repeat until convergence:
   - For each state $s$:
     $$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a) [r(s,a,s') + \gamma V_k(s')]$$
3. Return $V^\pi$

### Implementation

In [ ]:
@dataclass
class MDP:
    """Markov Decision Process definition."""
    n_states: int
    n_actions: int
    P: np.ndarray  # Transition probabilities [s, a, s'] -> probability
    R: np.ndarray  # Rewards [s, a, s'] -> reward
    gamma: float   # Discount factor
    terminal_states: Optional[List[int]] = None
    
    def __post_init__(self):
        """Validate MDP structure."""
        assert self.P.shape == (self.n_states, self.n_actions, self.n_states)
        assert self.R.shape == (self.n_states, self.n_actions, self.n_states)
        assert 0 <= self.gamma <= 1
        # Check probability normalization
        for s in range(self.n_states):
            for a in range(self.n_actions):
                prob_sum = np.sum(self.P[s, a, :])
                assert np.isclose(prob_sum, 1.0) or np.isclose(prob_sum, 0.0), \
                    f"Probabilities don't sum to 1 for state {s}, action {a}: {prob_sum}"
        
        if self.terminal_states is None:
            self.terminal_states = []


def policy_evaluation_iterative(
    mdp: MDP,
    policy: np.ndarray,
    theta: float = 1e-6,
    max_iterations: int = 10000
) -> Tuple[np.ndarray, int]:
    """
    Evaluate a policy using iterative policy evaluation.
    
    Args:
        mdp: Markov Decision Process
        policy: Policy array [n_states, n_actions] with action probabilities
        theta: Convergence threshold
        max_iterations: Maximum number of iterations
    
    Returns:
        V: State-value function [n_states]
        iterations: Number of iterations until convergence
    """
    V = np.zeros(mdp.n_states)
    
    for iteration in range(max_iterations):
        delta = 0
        
        # Update each state
        for s in range(mdp.n_states):
            # Skip terminal states
            if s in mdp.terminal_states:
                continue
                
            v = V[s]
            
            # Bellman expectation backup
            new_v = 0
            for a in range(mdp.n_actions):
                # Expected value under action a
                action_value = 0
                for s_prime in range(mdp.n_states):
                    prob = mdp.P[s, a, s_prime]
                    reward = mdp.R[s, a, s_prime]
                    action_value += prob * (reward + mdp.gamma * V[s_prime])
                
                # Weight by policy probability
                new_v += policy[s, a] * action_value
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        # Check convergence
        if delta < theta:
            return V, iteration + 1
    
    print(f"Warning: Max iterations ({max_iterations}) reached without convergence")
    return V, max_iterations


def policy_evaluation_analytical(
    mdp: MDP,
    policy: np.ndarray
) -> np.ndarray:
    """
    Evaluate a policy analytically by solving the linear system.
    
    Solves: V = (I - γP_π)^(-1) R_π
    
    Args:
        mdp: Markov Decision Process
        policy: Policy array [n_states, n_actions]
    
    Returns:
        V: State-value function [n_states]
    """
    n = mdp.n_states
    
    # Construct P_π and R_π
    P_pi = np.zeros((n, n))
    R_pi = np.zeros(n)
    
    for s in range(n):
        for a in range(mdp.n_actions):
            for s_prime in range(n):
                P_pi[s, s_prime] += policy[s, a] * mdp.P[s, a, s_prime]
                R_pi[s] += policy[s, a] * mdp.P[s, a, s_prime] * mdp.R[s, a, s_prime]
    
    # Solve V = (I - γP_π)^(-1) R_π
    I = np.eye(n)
    V = np.linalg.solve(I - mdp.gamma * P_pi, R_pi)
    
    return V


print("✅ Policy evaluation functions defined")

### Testing Policy Evaluation

Let's test both methods on a simple MDP.

In [ ]:
# Create a simple 3-state MDP
# States: 0 (start), 1 (intermediate), 2 (goal)
# Actions: 0 (left), 1 (right)

n_states = 3
n_actions = 2

# Transition probabilities
P = np.zeros((n_states, n_actions, n_states))

# State 0
P[0, 0, 0] = 1.0  # Left: stay
P[0, 1, 1] = 1.0  # Right: go to state 1

# State 1
P[1, 0, 0] = 1.0  # Left: go to state 0
P[1, 1, 2] = 1.0  # Right: go to goal

# State 2 (terminal)
P[2, 0, 2] = 1.0
P[2, 1, 2] = 1.0

# Rewards
R = np.zeros((n_states, n_actions, n_states))
R[1, 1, 2] = 1.0  # Reward for reaching goal

# Create MDP
simple_mdp = MDP(
    n_states=n_states,
    n_actions=n_actions,
    P=P,
    R=R,
    gamma=0.9,
    terminal_states=[2]
)

# Define a random policy
random_policy = np.ones((n_states, n_actions)) / n_actions

# Define an optimal policy (always go right)
optimal_policy = np.zeros((n_states, n_actions))
optimal_policy[:, 1] = 1.0  # Always choose action 1 (right)

print("Testing Policy Evaluation on Simple MDP")
print("=" * 50)

# Evaluate random policy
print("\n1. Random Policy (50% left, 50% right):")
V_random_iter, iters_random = policy_evaluation_iterative(simple_mdp, random_policy)
V_random_anal = policy_evaluation_analytical(simple_mdp, random_policy)

print(f"   Iterative method: {V_random_iter} (converged in {iters_random} iterations)")
print(f"   Analytical method: {V_random_anal}")
print(f"   Difference: {np.max(np.abs(V_random_iter - V_random_anal)):.2e}")

# Evaluate optimal policy
print("\n2. Optimal Policy (always go right):")
V_optimal_iter, iters_optimal = policy_evaluation_iterative(simple_mdp, optimal_policy)
V_optimal_anal = policy_evaluation_analytical(simple_mdp, optimal_policy)

print(f"   Iterative method: {V_optimal_iter} (converged in {iters_optimal} iterations)")
print(f"   Analytical method: {V_optimal_anal}")
print(f"   Difference: {np.max(np.abs(V_optimal_iter - V_optimal_anal)):.2e}")

print("\n✅ Policy evaluation methods match!")

## 3. Policy Iteration <a id='policy-iteration'></a>

### Theory

Policy iteration alternates between:
1. **Policy Evaluation**: Compute $V^\pi$
2. **Policy Improvement**: Create greedy policy $\pi'$ with respect to $V^\pi$

**Policy Improvement Theorem**: If $\pi'$ is greedy with respect to $V^\pi$, then $V^{\pi'}(s) \geq V^\pi(s)$ for all states.

**Greedy Policy:**
$$\pi'(s) = \arg\max_a \sum_{s'} p(s'|s,a)[r(s,a,s') + \gamma V^\pi(s')]$$

### Implementation

In [ ]:
def compute_action_values(
    mdp: MDP,
    V: np.ndarray,
    state: int
) -> np.ndarray:
    """
    Compute Q(s, a) for all actions in a given state.
    
    Q(s, a) = Σ_{s'} p(s'|s,a) [r(s,a,s') + γV(s')]
    
    Args:
        mdp: Markov Decision Process
        V: State-value function
        state: Current state
    
    Returns:
        Q: Action-value function [n_actions]
    """
    Q = np.zeros(mdp.n_actions)
    
    for a in range(mdp.n_actions):
        for s_prime in range(mdp.n_states):
            prob = mdp.P[state, a, s_prime]
            reward = mdp.R[state, a, s_prime]
            Q[a] += prob * (reward + mdp.gamma * V[s_prime])
    
    return Q


def policy_improvement(
    mdp: MDP,
    V: np.ndarray
) -> np.ndarray:
    """
    Create greedy policy with respect to value function.
    
    Args:
        mdp: Markov Decision Process
        V: State-value function
    
    Returns:
        policy: Deterministic policy [n_states, n_actions]
    """
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    
    for s in range(mdp.n_states):
        # Compute action values
        Q = compute_action_values(mdp, V, s)
        
        # Greedy action selection
        best_action = np.argmax(Q)
        policy[s, best_action] = 1.0
    
    return policy


def policy_iteration(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 100
) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Find optimal policy using policy iteration.
    
    Args:
        mdp: Markov Decision Process
        theta: Convergence threshold for policy evaluation
        max_iterations: Maximum number of policy iterations
    
    Returns:
        policy: Optimal policy [n_states, n_actions]
        V: Optimal value function [n_states]
        iterations: Number of policy iterations
    """
    # Initialize with random policy
    policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions
    
    for iteration in range(max_iterations):
        # Policy Evaluation
        V, _ = policy_evaluation_iterative(mdp, policy, theta)
        
        # Policy Improvement
        new_policy = policy_improvement(mdp, V)
        
        # Check if policy is stable
        if np.array_equal(policy, new_policy):
            return new_policy, V, iteration + 1
        
        policy = new_policy
    
    print(f"Warning: Max iterations ({max_iterations}) reached")
    return policy, V, max_iterations


print("✅ Policy iteration functions defined")

### Testing Policy Iteration

In [ ]:
print("Testing Policy Iteration on Simple MDP")
print("=" * 50)

# Run policy iteration
optimal_policy_pi, V_pi, iterations_pi = policy_iteration(simple_mdp)

print(f"\nConverged in {iterations_pi} policy iterations")
print(f"\nOptimal Value Function: {V_pi}")
print(f"\nOptimal Policy:")
for s in range(simple_mdp.n_states):
    action = np.argmax(optimal_policy_pi[s])
    action_name = "LEFT" if action == 0 else "RIGHT"
    print(f"  State {s}: {action_name} (V={V_pi[s]:.4f})")

print("\n✅ Policy iteration successful!")

## 4. Value Iteration <a id='value-iteration'></a>

### Theory

Value iteration combines policy evaluation and improvement in a single update.

**Bellman Optimality Equation:**
$$V_{k+1}(s) = \max_a \sum_{s'} p(s'|s,a)[r(s,a,s') + \gamma V_k(s')]$$

**Algorithm:**
1. Initialize $V(s) = 0$ for all states
2. Repeat until convergence:
   - For each state: $V(s) \leftarrow \max_a \sum_{s'} p(s'|s,a)[r + \gamma V(s')]$
3. Extract greedy policy from $V^*$

### Implementation

In [ ]:
def value_iteration(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 10000
) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Find optimal policy using value iteration.
    
    Args:
        mdp: Markov Decision Process
        theta: Convergence threshold
        max_iterations: Maximum number of iterations
    
    Returns:
        policy: Optimal policy [n_states, n_actions]
        V: Optimal value function [n_states]
        iterations: Number of iterations
    """
    V = np.zeros(mdp.n_states)
    
    for iteration in range(max_iterations):
        delta = 0
        
        # Update each state
        for s in range(mdp.n_states):
            # Skip terminal states
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            
            # Bellman optimality backup
            Q = compute_action_values(mdp, V, s)
            V[s] = np.max(Q)
            
            delta = max(delta, abs(v - V[s]))
        
        # Check convergence
        if delta < theta:
            # Extract optimal policy
            policy = policy_improvement(mdp, V)
            return policy, V, iteration + 1
    
    print(f"Warning: Max iterations ({max_iterations}) reached")
    policy = policy_improvement(mdp, V)
    return policy, V, max_iterations


print("✅ Value iteration function defined")

### Testing Value Iteration

In [ ]:
print("Testing Value Iteration on Simple MDP")
print("=" * 50)

# Run value iteration
optimal_policy_vi, V_vi, iterations_vi = value_iteration(simple_mdp)

print(f"\nConverged in {iterations_vi} iterations")
print(f"\nOptimal Value Function: {V_vi}")
print(f"\nOptimal Policy:")
for s in range(simple_mdp.n_states):
    action = np.argmax(optimal_policy_vi[s])
    action_name = "LEFT" if action == 0 else "RIGHT"
    print(f"  State {s}: {action_name} (V={V_vi[s]:.4f})")

# Compare with policy iteration
print(f"\n\nComparison with Policy Iteration:")
print(f"  Value difference: {np.max(np.abs(V_vi - V_pi)):.2e}")
print(f"  Policy match: {np.array_equal(optimal_policy_vi, optimal_policy_pi)}")
print(f"  Iterations: VI={iterations_vi}, PI={iterations_pi}")

print("\n✅ Value iteration successful!")

## 5. FrozenLake Environment <a id='frozenlake'></a>

Now let's apply our algorithms to a real Gymnasium environment: FrozenLake.

**FrozenLake Description:**
- 4x4 grid world
- Start at top-left (S)
- Goal at bottom-right (G)
- Holes (H) cause failure
- Frozen surfaces (F) are safe
- Slippery: actions are stochastic

In [ ]:
def extract_mdp_from_gym(env: gym.Env) -> MDP:
    """
    Extract MDP structure from a Gymnasium environment.
    
    Args:
        env: Gymnasium environment (must have discrete state/action spaces)
    
    Returns:
        mdp: MDP representation
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    # Initialize transition and reward arrays
    P = np.zeros((n_states, n_actions, n_states))
    R = np.zeros((n_states, n_actions, n_states))
    
    # Extract from environment's P dictionary
    for state in range(n_states):
        for action in range(n_actions):
            transitions = env.unwrapped.P[state][action]
            for prob, next_state, reward, done in transitions:
                P[state, action, next_state] += prob
                R[state, action, next_state] = reward
    
    # Find terminal states (states with only self-loops)
    terminal_states = []
    for s in range(n_states):
        is_terminal = True
        for a in range(n_actions):
            if not np.isclose(P[s, a, s], 1.0):
                is_terminal = False
                break
        if is_terminal:
            terminal_states.append(s)
    
    mdp = MDP(
        n_states=n_states,
        n_actions=n_actions,
        P=P,
        R=R,
        gamma=0.99,
        terminal_states=terminal_states
    )
    
    return mdp


# Create FrozenLake environment
env = gym.make('FrozenLake-v1', is_slippery=True, render_mode=None)
frozenlake_mdp = extract_mdp_from_gym(env)

print("FrozenLake MDP:")
print(f"  States: {frozenlake_mdp.n_states}")
print(f"  Actions: {frozenlake_mdp.n_actions}")
print(f"  Terminal states: {frozenlake_mdp.terminal_states}")
print(f"  Discount factor: {frozenlake_mdp.gamma}")

# Render the environment description
print("\nEnvironment Layout:")
env.reset()
print(env.unwrapped.desc.astype(str))
print("\nLegend: S=Start, F=Frozen, H=Hole, G=Goal")
print("Actions: 0=LEFT, 1=DOWN, 2=RIGHT, 3=UP")

print("\n✅ FrozenLake MDP extracted")

### Solve FrozenLake with Policy Iteration

In [ ]:
print("Solving FrozenLake with Policy Iteration")
print("=" * 50)

start_time = time.time()
policy_fl_pi, V_fl_pi, iters_fl_pi = policy_iteration(frozenlake_mdp)
time_pi = time.time() - start_time

print(f"\nConverged in {iters_fl_pi} policy iterations ({time_pi:.4f}s)")
print(f"\nOptimal Value Function (reshaped to 4x4):")
print(V_fl_pi.reshape(4, 4))

print(f"\nOptimal Policy (reshaped to 4x4):")
action_symbols = ['←', '↓', '→', '↑']
policy_grid = np.array([[action_symbols[np.argmax(policy_fl_pi[i*4 + j])] 
                        for j in range(4)] for i in range(4)])
print(policy_grid)

print("\n✅ FrozenLake solved with Policy Iteration!")

### Solve FrozenLake with Value Iteration

In [ ]:
print("Solving FrozenLake with Value Iteration")
print("=" * 50)

start_time = time.time()
policy_fl_vi, V_fl_vi, iters_fl_vi = value_iteration(frozenlake_mdp)
time_vi = time.time() - start_time

print(f"\nConverged in {iters_fl_vi} iterations ({time_vi:.4f}s)")
print(f"\nOptimal Value Function (reshaped to 4x4):")
print(V_fl_vi.reshape(4, 4))

print(f"\nOptimal Policy (reshaped to 4x4):")
policy_grid_vi = np.array([[action_symbols[np.argmax(policy_fl_vi[i*4 + j])] 
                           for j in range(4)] for i in range(4)])
print(policy_grid_vi)

print(f"\n\nComparison:")
print(f"  Value difference: {np.max(np.abs(V_fl_vi - V_fl_pi)):.2e}")
print(f"  Policy match: {np.array_equal(policy_fl_vi, policy_fl_pi)}")
print(f"  Time: PI={time_pi:.4f}s, VI={time_vi:.4f}s")
print(f"  Iterations: PI={iters_fl_pi}, VI={iters_fl_vi}")

print("\n✅ FrozenLake solved with Value Iteration!")

### Evaluate the Optimal Policy

In [ ]:
def evaluate_policy_in_env(
    env: gym.Env,
    policy: np.ndarray,
    n_episodes: int = 1000,
    max_steps: int = 100
) -> Tuple[float, float]:
    """
    Evaluate a policy by running episodes in the environment.
    
    Args:
        env: Gymnasium environment
        policy: Policy array [n_states, n_actions]
        n_episodes: Number of episodes to run
        max_steps: Maximum steps per episode
    
    Returns:
        success_rate: Fraction of episodes reaching goal
        avg_reward: Average total reward per episode
    """
    successes = 0
    total_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        episode_reward = 0
        
        for step in range(max_steps):
            # Select action according to policy
            action = np.argmax(policy[state])
            state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            
            if terminated or truncated:
                if reward > 0:  # Reached goal
                    successes += 1
                break
        
        total_rewards.append(episode_reward)
    
    success_rate = successes / n_episodes
    avg_reward = np.mean(total_rewards)
    
    return success_rate, avg_reward


print("Evaluating Optimal Policy in FrozenLake")
print("=" * 50)

success_rate, avg_reward = evaluate_policy_in_env(env, policy_fl_vi, n_episodes=1000)

print(f"\nResults over 1000 episodes:")
print(f"  Success rate: {success_rate:.2%}")
print(f"  Average reward: {avg_reward:.4f}")

print("\n✅ Policy evaluation complete!")

## 6. GridWorld Solutions <a id='gridworld'></a>

Let's create a custom GridWorld with obstacles and solve it.

In [ ]:
class GridWorldMDP:
    """
    Custom GridWorld environment with obstacles.
    """
    def __init__(
        self,
        height: int = 5,
        width: int = 5,
        start: Tuple[int, int] = (0, 0),
        goal: Tuple[int, int] = (4, 4),
        obstacles: Optional[List[Tuple[int, int]]] = None,
        gamma: float = 0.9,
        step_cost: float = -0.01
    ):
        self.height = height
        self.width = width
        self.start = start
        self.goal = goal
        self.obstacles = obstacles if obstacles else []
        self.step_cost = step_cost
        
        # Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
        self.n_actions = 4
        self.action_effects = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        
        # Create state mapping
        self.state_to_pos = {}
        self.pos_to_state = {}
        state_idx = 0
        
        for i in range(height):
            for j in range(width):
                if (i, j) not in self.obstacles:
                    self.state_to_pos[state_idx] = (i, j)
                    self.pos_to_state[(i, j)] = state_idx
                    state_idx += 1
        
        self.n_states = state_idx
        self.start_state = self.pos_to_state[start]
        self.goal_state = self.pos_to_state[goal]
        
        # Build MDP
        self.mdp = self._build_mdp(gamma)
    
    def _build_mdp(self, gamma: float) -> MDP:
        """Build MDP transition and reward matrices."""
        P = np.zeros((self.n_states, self.n_actions, self.n_states))
        R = np.zeros((self.n_states, self.n_actions, self.n_states))
        
        for state in range(self.n_states):
            pos = self.state_to_pos[state]
            
            # Terminal state
            if state == self.goal_state:
                for action in range(self.n_actions):
                    P[state, action, state] = 1.0
                    R[state, action, state] = 0.0
                continue
            
            # Non-terminal states
            for action in range(self.n_actions):
                dy, dx = self.action_effects[action]
                new_pos = (pos[0] + dy, pos[1] + dx)
                
                # Check boundaries and obstacles
                if (0 <= new_pos[0] < self.height and 
                    0 <= new_pos[1] < self.width and 
                    new_pos not in self.obstacles):
                    new_state = self.pos_to_state[new_pos]
                else:
                    new_state = state  # Stay in place
                
                P[state, action, new_state] = 1.0
                
                # Rewards
                if new_state == self.goal_state:
                    R[state, action, new_state] = 1.0
                else:
                    R[state, action, new_state] = self.step_cost
        
        return MDP(
            n_states=self.n_states,
            n_actions=self.n_actions,
            P=P,
            R=R,
            gamma=gamma,
            terminal_states=[self.goal_state]
        )
    
    def render_policy(self, policy: np.ndarray) -> np.ndarray:
        """Render policy as a grid."""
        action_symbols = ['↑', '↓', '←', '→']
        grid = np.full((self.height, self.width), ' ', dtype=object)
        
        for state in range(self.n_states):
            pos = self.state_to_pos[state]
            action = np.argmax(policy[state])
            grid[pos] = action_symbols[action]
        
        # Mark special positions
        grid[self.start] = 'S'
        grid[self.goal] = 'G'
        for obs in self.obstacles:
            grid[obs] = '█'
        
        return grid
    
    def render_values(self, V: np.ndarray) -> np.ndarray:
        """Render value function as a grid."""
        grid = np.full((self.height, self.width), np.nan)
        
        for state in range(self.n_states):
            pos = self.state_to_pos[state]
            grid[pos] = V[state]
        
        return grid


# Create GridWorld with obstacles
obstacles = [(1, 1), (1, 2), (2, 2), (3, 1)]
gridworld = GridWorldMDP(
    height=5,
    width=5,
    start=(0, 0),
    goal=(4, 4),
    obstacles=obstacles,
    gamma=0.9,
    step_cost=-0.01
)

print("GridWorld Created:")
print(f"  Size: {gridworld.height}x{gridworld.width}")
print(f"  States: {gridworld.n_states}")
print(f"  Start: {gridworld.start}")
print(f"  Goal: {gridworld.goal}")
print(f"  Obstacles: {len(obstacles)}")

print("\n✅ GridWorld MDP created")

### Solve GridWorld

In [ ]:
print("Solving GridWorld with Value Iteration")
print("=" * 50)

# Solve with value iteration
policy_gw, V_gw, iters_gw = value_iteration(gridworld.mdp)

print(f"\nConverged in {iters_gw} iterations")

# Render results
print("\nOptimal Policy:")
policy_grid = gridworld.render_policy(policy_gw)
for row in policy_grid:
    print('  ' + ' '.join(row))

print("\nOptimal Value Function:")
value_grid = gridworld.render_values(V_gw)
print(value_grid)

print("\n✅ GridWorld solved!")

## 7. Visualization and Analysis <a id='visualization'></a>

Let's create comprehensive visualizations of our solutions.

In [ ]:
def plot_value_function(
    V: np.ndarray,
    shape: Tuple[int, int],
    title: str = "Value Function",
    ax: Optional[plt.Axes] = None
) -> plt.Axes:
    """
    Plot value function as a heatmap.
    
    Args:
        V: Value function array
        shape: Grid shape (height, width)
        title: Plot title
        ax: Matplotlib axes (optional)
    
    Returns:
        ax: Matplotlib axes
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    V_grid = V.reshape(shape)
    
    im = ax.imshow(V_grid, cmap='viridis', aspect='auto')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Column', fontsize=12)
    ax.set_ylabel('Row', fontsize=12)
    
    # Add value annotations
    for i in range(shape[0]):
        for j in range(shape[1]):
            text = ax.text(j, i, f'{V_grid[i, j]:.3f}',
                          ha="center", va="center", color="w", fontsize=10)
    
    plt.colorbar(im, ax=ax, label='Value')
    
    return ax


def plot_policy(
    policy: np.ndarray,
    shape: Tuple[int, int],
    title: str = "Policy",
    ax: Optional[plt.Axes] = None
) -> plt.Axes:
    """
    Plot policy as arrows.
    
    Args:
        policy: Policy array [n_states, n_actions]
        shape: Grid shape (height, width)
        title: Plot title
        ax: Matplotlib axes (optional)
    
    Returns:
        ax: Matplotlib axes
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    action_to_arrow = {
        0: '←',  # LEFT
        1: '↓',  # DOWN
        2: '→',  # RIGHT
        3: '↑'   # UP
    }
    
    # Create grid
    grid = np.zeros(shape)
    ax.imshow(grid, cmap='gray_r', alpha=0.3, aspect='auto')
    
    # Add arrows
    for state in range(policy.shape[0]):
        i = state // shape[1]
        j = state % shape[1]
        action = np.argmax(policy[state])
        arrow = action_to_arrow.get(action, '?')
        ax.text(j, i, arrow, ha='center', va='center', fontsize=20, color='blue')
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Column', fontsize=12)
    ax.set_ylabel('Row', fontsize=12)
    ax.set_xticks(range(shape[1]))
    ax.set_yticks(range(shape[0]))
    ax.grid(True, alpha=0.3)
    
    return ax


# Visualize FrozenLake solution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_value_function(V_fl_vi, (4, 4), "FrozenLake: Optimal Value Function", axes[0])
plot_policy(policy_fl_vi, (4, 4), "FrozenLake: Optimal Policy", axes[1])

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/1b_frozenlake_solution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization created!")

### Convergence Analysis

In [ ]:
def value_iteration_with_history(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 10000
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """
    Value iteration that tracks convergence history.
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        deltas: List of max value changes per iteration
    """
    V = np.zeros(mdp.n_states)
    deltas = []
    
    for iteration in range(max_iterations):
        delta = 0
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            Q = compute_action_values(mdp, V, s)
            V[s] = np.max(Q)
            delta = max(delta, abs(v - V[s]))
        
        deltas.append(delta)
        
        if delta < theta:
            policy = policy_improvement(mdp, V)
            return policy, V, deltas
    
    policy = policy_improvement(mdp, V)
    return policy, V, deltas


# Run with history tracking
print("Analyzing Convergence Behavior")
print("=" * 50)

_, _, deltas_fl = value_iteration_with_history(frozenlake_mdp)
_, _, deltas_gw = value_iteration_with_history(gridworld.mdp)

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(deltas_fl, label='FrozenLake', linewidth=2, marker='o', markersize=4, alpha=0.7)
ax.plot(deltas_gw, label='GridWorld', linewidth=2, marker='s', markersize=4, alpha=0.7)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Max Value Change (Δ)', fontsize=12)
ax.set_title('Value Iteration Convergence', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/1b_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFrozenLake converged in {len(deltas_fl)} iterations")
print(f"GridWorld converged in {len(deltas_gw)} iterations")

print("\n✅ Convergence analysis complete!")

## 8. Performance Comparison <a id='comparison'></a>

Let's compare policy iteration and value iteration systematically.

In [ ]:
def benchmark_algorithms(
    mdp: MDP,
    name: str
) -> Dict[str, any]:
    """
    Benchmark both policy iteration and value iteration.
    
    Args:
        mdp: MDP to solve
        name: Name for display
    
    Returns:
        results: Dictionary with timing and iteration counts
    """
    results = {'name': name}
    
    # Policy Iteration
    start = time.time()
    policy_pi, V_pi, iters_pi = policy_iteration(mdp)
    time_pi = time.time() - start
    
    results['pi_time'] = time_pi
    results['pi_iters'] = iters_pi
    results['pi_value'] = V_pi
    results['pi_policy'] = policy_pi
    
    # Value Iteration
    start = time.time()
    policy_vi, V_vi, iters_vi = value_iteration(mdp)
    time_vi = time.time() - start
    
    results['vi_time'] = time_vi
    results['vi_iters'] = iters_vi
    results['vi_value'] = V_vi
    results['vi_policy'] = policy_vi
    
    # Comparison
    results['value_diff'] = np.max(np.abs(V_pi - V_vi))
    results['policy_match'] = np.array_equal(policy_pi, policy_vi)
    
    return results


print("Comprehensive Algorithm Benchmark")
print("=" * 70)

# Benchmark on multiple environments
benchmarks = [
    benchmark_algorithms(simple_mdp, "Simple 3-State"),
    benchmark_algorithms(frozenlake_mdp, "FrozenLake 4x4"),
    benchmark_algorithms(gridworld.mdp, "GridWorld 5x5")
]

# Display results
print(f"\n{'Environment':<20} {'Algorithm':<15} {'Time (s)':<12} {'Iterations':<12}")
print("-" * 70)

for b in benchmarks:
    print(f"{b['name']:<20} {'Policy Iter':<15} {b['pi_time']:<12.6f} {b['pi_iters']:<12}")
    print(f"{'':<20} {'Value Iter':<15} {b['vi_time']:<12.6f} {b['vi_iters']:<12}")
    print(f"{'':<20} {'Match':<15} {str(b['policy_match']):<12} {'':<12}")
    print("-" * 70)

print("\n✅ Benchmark complete!")

### Visualize Benchmark Results

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

env_names = [b['name'] for b in benchmarks]
pi_times = [b['pi_time'] for b in benchmarks]
vi_times = [b['vi_time'] for b in benchmarks]
pi_iters = [b['pi_iters'] for b in benchmarks]
vi_iters = [b['vi_iters'] for b in benchmarks]

x = np.arange(len(env_names))
width = 0.35

# Time comparison
axes[0].bar(x - width/2, pi_times, width, label='Policy Iteration', alpha=0.8)
axes[0].bar(x + width/2, vi_times, width, label='Value Iteration', alpha=0.8)
axes[0].set_xlabel('Environment', fontsize=12)
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('Execution Time Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(env_names, rotation=15, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Iteration comparison
axes[1].bar(x - width/2, pi_iters, width, label='Policy Iteration', alpha=0.8)
axes[1].bar(x + width/2, vi_iters, width, label='Value Iteration', alpha=0.8)
axes[1].set_xlabel('Environment', fontsize=12)
axes[1].set_ylabel('Iterations', fontsize=12)
axes[1].set_title('Iteration Count Comparison', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(env_names, rotation=15, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/1b_algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Comparison visualization complete!")

## 9. Exercises <a id='exercises'></a>

### Exercise 1: Policy Evaluation Variants

Implement in-place policy evaluation, where state values are updated immediately rather than simultaneously.

**Hint**: Use only one array instead of old and new value arrays.

In [ ]:
def policy_evaluation_inplace(
    mdp: MDP,
    policy: np.ndarray,
    theta: float = 1e-6
) -> Tuple[np.ndarray, int]:
    """
    In-place policy evaluation.
    
    TODO: Implement this function
    """
    # YOUR CODE HERE
    pass

# Test your implementation
# V_inplace, iters = policy_evaluation_inplace(simple_mdp, random_policy)
# print(f"In-place converged in {iters} iterations")
# print(f"Value function: {V_inplace}")

### Exercise 2: Asynchronous Value Iteration

Implement asynchronous value iteration where states are updated in random order rather than sequentially.

**Question**: Does this affect convergence? Run experiments to find out.

In [ ]:
def value_iteration_async(
    mdp: MDP,
    theta: float = 1e-6
) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Asynchronous value iteration with random state ordering.
    
    TODO: Implement this function
    """
    # YOUR CODE HERE
    pass

# Test and compare with synchronous version
# policy_async, V_async, iters_async = value_iteration_async(frozenlake_mdp)
# print(f"Async VI: {iters_async} iterations")
# print(f"Sync VI: {iters_fl_vi} iterations")

### Exercise 3: Modified Policy Iteration

Implement modified policy iteration that performs only k steps of policy evaluation (rather than converging fully) before policy improvement.

**Question**: How does the value of k affect convergence and performance?

In [ ]:
def modified_policy_iteration(
    mdp: MDP,
    k: int = 10,
    theta: float = 1e-6
) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Modified policy iteration with k evaluation steps.
    
    TODO: Implement this function
    
    Args:
        k: Number of evaluation steps per iteration
    """
    # YOUR CODE HERE
    pass

# Experiment with different k values
# for k in [1, 5, 10, 50]:
#     policy, V, iters = modified_policy_iteration(frozenlake_mdp, k=k)
#     print(f"k={k}: {iters} iterations")

### Exercise 4: Larger GridWorld

Create and solve a 10x10 GridWorld with random obstacles.

**Tasks**:
1. Generate random obstacles (20% of cells)
2. Ensure path exists from start to goal
3. Solve with both algorithms
4. Visualize the solution
5. Compare performance

In [ ]:
# TODO: Create and solve larger GridWorld
# YOUR CODE HERE

# Hints:
# - Use np.random to generate obstacle positions
# - Check connectivity with BFS/DFS
# - Use the GridWorldMDP class defined above
# - Use the visualization functions

### Exercise 5: FrozenLake 8x8

Solve the larger FrozenLake-v1 environment (8x8 grid).

**Questions**:
1. How do iteration counts scale?
2. What is the success rate of the optimal policy?
3. How does the slippery parameter affect difficulty?

In [ ]:
# TODO: Solve FrozenLake-v1 8x8
# YOUR CODE HERE

# env_8x8 = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)
# mdp_8x8 = extract_mdp_from_gym(env_8x8)
# ...

# Try both is_slippery=True and is_slippery=False

---

## Summary

In this notebook, you have:

1. ✅ **Implemented Policy Evaluation** - Both iterative and analytical methods
2. ✅ **Implemented Policy Iteration** - Complete algorithm with evaluation and improvement
3. ✅ **Implemented Value Iteration** - Combined Bellman optimality updates
4. ✅ **Solved Real Environments** - FrozenLake and custom GridWorld
5. ✅ **Created Visualizations** - Value functions, policies, and convergence
6. ✅ **Compared Algorithms** - Performance analysis and benchmarking

### Key Takeaways

- **Policy Iteration**: Alternates evaluation and improvement, converges in fewer iterations
- **Value Iteration**: Combines both steps, often faster in wall-clock time
- **Both methods guarantee convergence** to optimal policy for finite MDPs
- **Dynamic programming requires full model knowledge** (P and R)
- **Trade-offs exist** between iteration count and computational cost per iteration

### Next Steps

In **Lesson 2**, we will:
- Dive deeper into dynamic programming theory
- Explore generalized policy iteration
- Study asynchronous and prioritized sweeping methods
- Prepare for model-free methods (when P and R are unknown)

---

**Lesson 1b Complete!** 🎉

You now have production-ready implementations of fundamental dynamic programming algorithms for solving MDPs.